In [1]:
from fastDSA.simdist import SimDistConfig, FastDSASimilarity
import numpy as np
from scipy.integrate import solve_ivp

In [2]:
use_synthetic = True  # <-- set to False to load from folders

folder_A = "/home/behradarm/Documents/fastDSA/fastDSA_EEG/out/post_stimulus/"   # folder of .npy files for Dataset A (if not synthetic)
folder_B = "/home/behradarm/Documents/fastDSA/fastDSA_EEG/out/pre_stimulus/"   # folder of .npy files for Dataset B (if not synthetic)

def load_trajs_from_folder(folder: str):
    """Load all .npy files in a folder as a list of arrays (T, n_features).\"\"\"\"\"\""""
    path = Path(folder)
    if not path.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")
    trajs = []
    for f in sorted(path.glob("*.npy")):
        arr = np.load(f, allow_pickle=False)
        if arr.ndim != 2:
            raise ValueError(f'File {f} must be 2D (T, n_features). Got shape {arr.shape}.' )
        trajs.append(arr)
    if len(trajs) == 0:
        raise ValueError(f'No .npy files found in {folder}')
    return trajs

# --- Synthetic demo systems (2D) ---
def system_a(t, state, eps=0.5):
    x, y = state
    dx = -1.0 * x + eps * x * y
    dy = -2.0 * y - eps * x**2
    return [dx, dy]

def system_b(t, state, eps=0.5):
    x, y = state
    dx = -1.5 * x + 0.5 * y - eps * x**2
    dy = 0.5 * x - 1.5 * y - eps * y**2
    # dx = -1.0 * x + eps * x * y
    # dy = -3.0 * y - eps * x**2
    return [dx, dy]

def generate_trajectory(system, x0, T=10.0, dt=0.01, eps=0.5):
    t_eval = np.arange(0, T, dt)
    sol = solve_ivp(system, (0, T), x0, t_eval=t_eval, args=(eps,))
    return sol.y.T  # (T, 2)

if use_synthetic:
    # Build small demo sets
    num_per_set = 10
    T=10.0; dt=0.01
    eps=0.5
    rng = np.random.default_rng(42)

    dataset_A = [generate_trajectory(system_a, rng.uniform(-4,4,size=2), T=T, dt=dt, eps=eps) for _ in range(num_per_set)]
    dataset_B = [generate_trajectory(system_b, rng.uniform(-4,4,size=2), T=T, dt=dt, eps=eps) for _ in range(num_per_set)]
else:
    dataset_A = load_trajs_from_folder(folder_A)
    dataset_B = load_trajs_from_folder(folder_B)

print(f"Loaded {len(dataset_A)} trajectories for A, {len(dataset_B)} for B.")
print(f'Example shape A[0]: {dataset_A[0].shape}.')

Loaded 10 trajectories for A, 10 for B.
Example shape A[0]: (1000, 2).


In [3]:
import numpy as np
import torch
from time import time

# package imports
from fastDSA.simdist import SimDistConfig, FastDSASimilarity


def to_channels_time(traj_Tn: np.ndarray) -> np.ndarray:
    """
    Convert (T, n_features) -> (channels, timepoints).
    Your package expects (channels, timepoints).
    """
    if traj_Tn.ndim != 2:
        raise ValueError(f"Trajectory must be 2D (T, n_features). Got {traj_Tn.shape}")
    return traj_Tn.T


# Convert your datasets
dataset_A_ct = [to_channels_time(tr) for tr in dataset_A]
dataset_B_ct = [to_channels_time(tr) for tr in dataset_B]

print("A example (C,T):", dataset_A_ct[0].shape)
print("B example (C,T):", dataset_B_ct[0].shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


def run_one(method: str, iters=300, lr=1e-2, n_delays=20, delay_interval=1, rank=None):
    cfg = SimDistConfig(
        n_delays=n_delays,
        delay_interval=delay_interval,
        rank=rank,                 # None triggers SVHT global rank
        method=method,
        iters=iters,
        lr=lr,
        eta=None,                  # for land, defaults to lr in simdist
        gamma=0.98,
        n_Cmats=2,
        device=device,
        verbose=True,
    )
    sim = FastDSASimilarity(cfg)

    t0 = time()
    score, used_rank = sim.fit_score(dataset_A_ct, dataset_B_ct)
    dt = time() - t0

    print(f"\n=== method={method} ===")
    print("score:", score)
    print("used_rank:", used_rank)
    print("elapsed (s):", round(dt, 3))
    return score, used_rank


# Keep it light at first
scores = {}
for m in ["ro", "rim", "land"]:
    scores[m] = run_one(m, iters=200, lr=1e-2, n_delays=15, delay_interval=1, rank=None)

# Optional: kw (only if kooplearn import is fixed in kwdsa.py)
# scores["kw"] = run_one("kw", iters=0, lr=0, n_delays=15, delay_interval=1, rank=None)

print("\nDone. Scores:", scores)


A example (C,T): (2, 1000)
B example (C,T): (2, 1000)
Using device: cuda
[fastDSA] Detected global rank via SVHT: 14
Optimal C computed for group O(n) with regularization and low-rank approximation.

=== method=ro ===
score: 1.3456069231033325
used_rank: 14
elapsed (s): 1.523
[fastDSA] Detected global rank via SVHT: 14
[Landing] iter=1/200 loss=1.772424e+00
[Landing] iter=101/200 loss=1.009860e+00
[Landing] iter=200/200 loss=8.786054e-01
[Landing] score=0.975115, time=0.132s

=== method=rim ===
score: 0.9751150012016296
used_rank: 14
elapsed (s): 0.173
[fastDSA] Detected global rank via SVHT: 14


100%|██████████| 200/200 [00:00<00:00, 1731.80it/s]


=== method=land ===
score: 1.002195954322815
used_rank: 14
elapsed (s): 0.169

Done. Scores: {'ro': (1.3456069231033325, 14), 'rim': (0.9751150012016296, 14), 'land': (1.002195954322815, 14)}
